### And now - Week 3 Day 3

## AutoGen Core

Something a little different.

This is agnostic to the underlying Agent framework

You can use AutoGen AgentChat, or you can use something else; it's an Agent interaction framework.

From that point of view, it's positioned similarly to LangGraph.

### The fundamental principle

Autogen Core decouples an agent's logic from how messages are delivered.  
The framework provides a communication infrastructure, along with agent lifecycle, and the agents are responsible for their own work.

The communication infrastructure is called a Runtime.

There are 2 types: **Standalone** and **Distributed**.

Today we will use a standalone runtime: the **SingleThreadedAgentRuntime**, a local embedded agent runtime implementation.

Tomorrow we'll briefly look at a Distributed runtime.


In [1]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler
from autogen_core import SingleThreadedAgentRuntime
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.openai import OpenAIChatCompletionClient
from dotenv import load_dotenv

load_dotenv(override=True)


True

In [4]:
# Enable AutoGen's built-in tracing for monitoring agent interactions
# This provides detailed logs of agent decisions, tool calls, and message flows
import logging
from autogen_core import TRACE_LOGGER_NAME

# Set up logging to capture AutoGen traces
logging.basicConfig(level=logging.INFO)
trace_logger = logging.getLogger(TRACE_LOGGER_NAME)
trace_logger.setLevel(logging.DEBUG)

# Create a console handler for better formatting
console_handler = logging.StreamHandler()
console_handler.setLevel(logging.DEBUG)
formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
console_handler.setFormatter(formatter)
trace_logger.addHandler(console_handler)

print("AutoGen tracing enabled! Agent interactions will now be logged in detail.")

AutoGen tracing enabled! Agent interactions will now be logged in detail.


### First we define our Message object

Whatever structure we want for messages in our Agent framework.

In [5]:
# Let's have a simple one!

# In AutoGen Core, messages are the primary way agents communicate.
# This Message dataclass defines a simple message structure with just content.
# In a real application, you might add more fields like sender, timestamp, or metadata.
# The @dataclass decorator automatically generates __init__, __repr__, etc.

@dataclass
class Message:
    content: str

### Now we define our Agent

A subclass of RoutedAgent.

Every Agent has an **Agent ID** which has 2 components:  
`agent.id.type` describes the kind of agent it is  
`agent.id.key` gives it its unique identifier

Any method with the `@message_handler` decorated will have the opportunity to receive messages.


In [6]:
# SimpleAgent demonstrates the basic structure of an AutoGen Core agent.
# It inherits from RoutedAgent, which provides the foundation for message routing.
# The agent type is "Simple" (passed to super().__init__).

class SimpleAgent(RoutedAgent):
    def __init__(self) -> None:
        super().__init__("Simple")

    # The @message_handler decorator marks this method as one that can receive messages.
    # It takes a Message object and MessageContext, and returns a Message.
    # This agent simply acknowledges the message and disagrees with it.
    @message_handler
    async def on_my_message(self, message: Message, ctx: MessageContext) -> Message:
        # Access agent identity via self.id.type and self.id.key
        # self.id.type is "Simple", self.id.key will be set when registered (e.g., "default")
        return Message(content=f"This is {self.id.type}-{self.id.key}. You said '{message.content}' and I disagree.")

### OK let's create a Standalone runtime and register our agent type

In [8]:

runtime = SingleThreadedAgentRuntime()
await SimpleAgent.register(runtime, "simple_agent", lambda: SimpleAgent())

AgentType(type='simple_agent')

### Alright! Let's start a runtime and send a message

In [ ]:
runtime.start()

INFO:autogen_core:Calling message handler for simple_agent/default with message type Message sent by Unknown
INFO:autogen_core.events:{"payload": "{\"content\": \"Well hi there!\"}", "sender": null, "receiver": "simple_agent/default", "kind": "MessageKind.DIRECT", "delivery_stage": "DeliveryStage.DELIVER", "type": "Message"}
INFO:autogen_core.events:{"payload": "{\"content\": \"Well hi there!\"}", "sender": null, "receiver": "simple_agent/default", "kind": "MessageKind.DIRECT", "delivery_stage": "DeliveryStage.DELIVER", "type": "Message"}
INFO:autogen_core.events:{"payload": "{\"content\": \"This is simple_agent-default. You said 'Well hi there!' and I disagree.\"}", "sender": "simple_agent/default", "receiver": null, "kind": "MessageKind.RESPOND", "delivery_stage": "DeliveryStage.SEND", "type": "Message"}
INFO:autogen_core.events:{"payload": "{\"content\": \"This is simple_agent-default. You said 'Well hi there!' and I disagree.\"}", "sender": "simple_agent/default", "receiver": null,

In [10]:
agent_id = AgentId("simple_agent", "default")
response = await runtime.send_message(Message("Well hi there!"), agent_id)
print(">>>", response.content)

INFO:autogen_core.events:{"payload": "{\"content\": \"Well hi there!\"}", "sender": null, "receiver": "simple_agent/default", "kind": "MessageKind.DIRECT", "delivery_stage": "DeliveryStage.SEND", "type": "Message"}
INFO:autogen_core:Sending message of type Message to simple_agent: {'content': 'Well hi there!'}
INFO:autogen_core:Sending message of type Message to simple_agent: {'content': 'Well hi there!'}


>>> This is simple_agent-default. You said 'Well hi there!' and I disagree.


In [11]:
await runtime.stop()
await runtime.close()

In [12]:
from autogen_ext.models.ollama import OllamaChatCompletionClient
ollamamodel_client = OllamaChatCompletionClient(model="llama3.2")

### OK Now let's do something more interesting

We'll use an AgentChat Assistant!

In [13]:

class MyLLMAgent(RoutedAgent):
    def __init__(self) -> None:
        super().__init__("LLMAgent")
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini")
        self._delegate = AssistantAgent("LLMAgent", model_client=ollamamodel_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        print(f"{self.id.type} received message: {message.content}")
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        reply = response.chat_message.content
        print(f"{self.id.type} responded: {reply}")
        return Message(content=reply)
    


In [ ]:
# Create a new runtime instance for this multi-agent interaction
# This demonstrates how to set up multiple agent types in the same runtime
from autogen_core import SingleThreadedAgentRuntime

runtime = SingleThreadedAgentRuntime()

# Register agent types with the runtime
# Each agent type needs a unique name and a factory function (lambda) to create instances
# The factory function ensures each agent gets its own instance when needed
await SimpleAgent.register(runtime, "simple_agent", lambda: SimpleAgent())
await MyLLMAgent.register(runtime, "LLMAgent", lambda: MyLLMAgent())

AgentType(type='LLMAgent')

In [ ]:
# Start the runtime to begin processing messages asynchronously
runtime.start()  # Start processing messages in the background.

# Send initial message to the LLM agent
# This demonstrates agent-to-agent communication through the runtime
response = await runtime.send_message(Message("Hi there!"), AgentId("LLMAgent", "default"))
print(">>>", response.content)

# Send the LLM's response to the simple agent for processing
# Shows how agents can pass messages to each other
response =  await runtime.send_message(Message(response.content), AgentId("simple_agent", "default"))
print(">>>", response.content)

# Send the simple agent's response back to the LLM agent
# Creates a conversation loop between different agent types
response = await runtime.send_message(Message(response.content), AgentId("LLMAgent", "default"))

INFO:autogen_core.events:{"payload": "{\"content\": \"Hi there!\"}", "sender": null, "receiver": "LLMAgent/default", "kind": "MessageKind.DIRECT", "delivery_stage": "DeliveryStage.SEND", "type": "Message"}
INFO:autogen_core:Sending message of type Message to LLMAgent: {'content': 'Hi there!'}
INFO:autogen_core:Calling message handler for LLMAgent/default with message type Message sent by Unknown
INFO:autogen_core.events:{"payload": "{\"content\": \"Hi there!\"}", "sender": null, "receiver": "LLMAgent/default", "kind": "MessageKind.DIRECT", "delivery_stage": "DeliveryStage.DELIVER", "type": "Message"}
INFO:autogen_core:Sending message of type Message to LLMAgent: {'content': 'Hi there!'}
INFO:autogen_core:Calling message handler for LLMAgent/default with message type Message sent by Unknown
INFO:autogen_core.events:{"payload": "{\"content\": \"Hi there!\"}", "sender": null, "receiver": "LLMAgent/default", "kind": "MessageKind.DIRECT", "delivery_stage": "DeliveryStage.DELIVER", "type": "

LLMAgent received message: Hi there!


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:autogen_core.events:{"type": "LLMCall", "messages": [{"role": "system", "content": "You are a helpful AI assistant. Solve tasks using your tools. Reply with TERMINATE when the task has been completed.", "thinking": null, "images": null, "tool_calls": null}, {"role": "user", "content": "Hi there!", "thinking": null, "images": null, "tool_calls": null}], "response": {"model": "llama3.2", "created_at": "2025-10-20T21:46:03.76Z", "done": true, "done_reason": "stop", "total_duration": 8523094625, "load_duration": 3297198958, "prompt_eval_count": 52, "prompt_eval_duration": 2894317667, "eval_count": 24, "eval_duration": 2293648210, "message": {"role": "assistant", "content": "Hello! How can I assist you today? Do you have a specific task or question you'd like help with?", "thinking": null, "images": null, "tool_calls": null}}, "prompt_tokens": 52, "completion_tokens": 24, "agent_id": "LLMAgent/default"}
INF

LLMAgent responded: Hello! How can I assist you today? Do you have a specific task or question you'd like help with?
>>> Hello! How can I assist you today? Do you have a specific task or question you'd like help with?
>>> This is simple_agent-default. You said 'Hello! How can I assist you today? Do you have a specific task or question you'd like help with?' and I disagree.
LLMAgent received message: This is simple_agent-default. You said 'Hello! How can I assist you today? Do you have a specific task or question you'd like help with?' and I disagree.


INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
INFO:autogen_core.events:{"type": "LLMCall", "messages": [{"role": "system", "content": "You are a helpful AI assistant. Solve tasks using your tools. Reply with TERMINATE when the task has been completed.", "thinking": null, "images": null, "tool_calls": null}, {"role": "user", "content": "Hi there!", "thinking": null, "images": null, "tool_calls": null}, {"role": "assistant", "content": "Hello! How can I assist you today? Do you have a specific task or question you'd like help with?", "thinking": null, "images": null, "tool_calls": null}, {"role": "user", "content": "This is simple_agent-default. You said 'Hello! How can I assist you today? Do you have a specific task or question you'd like help with?' and I disagree.", "thinking": null, "images": null, "tool_calls": null}], "response": {"model": "llama3.2", "created_at": "2025-10-20T21:46:10.774482Z", "done": true, "done_reason": "stop", "total_duration"

LLMAgent responded: You're correct, I shouldn't have responded as if we were starting a conversation from scratch since I was already aware of the context. Let's proceed with your original request then. You had mentioned earlier that I should respond with "TERMINATE" when the task has been completed. What is the task you'd like me to assist with?


In [16]:
await runtime.stop()
await runtime.close()

### OK now let's show this at work - let's have 3 agents interact!

In [17]:
from autogen_ext.models.ollama import OllamaChatCompletionClient


class Player1Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)
    
class Player2Agent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OllamaChatCompletionClient(model="llama3.2", temperature=1.0)
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        text_message = TextMessage(content=message.content, source="user")
        response = await self._delegate.on_messages([text_message], ctx.cancellation_token)
        return Message(content=response.chat_message.content)

In [ ]:
# Define the judge's prompt template for determining the winner
JUDGE = "You are judging a game of rock, paper, scissors. The players have made these choices:\n"

class RockPaperScissorsAgent(RoutedAgent):
    def __init__(self, name: str) -> None:
        super().__init__(name)
        model_client = OpenAIChatCompletionClient(model="gpt-4o-mini", temperature=1.0)
        # The _delegate is an AssistantAgent that will act as the judge
        self._delegate = AssistantAgent(name, model_client=model_client)

    @message_handler
    async def handle_my_message_type(self, message: Message, ctx: MessageContext) -> Message:
        # Step 1: Create instruction for players to make their moves
        instruction = "You are playing rock, paper, scissors. Respond only with the one word, one of the following: rock, paper, or scissors."
        message = Message(content=instruction)

        # Step 2: Send the instruction to both players and get their responses
        inner_1 = AgentId("player1", "default")
        inner_2 = AgentId("player2", "default")
        response1 = await self.send_message(message, inner_1)  # Player 1's choice
        response2 = await self.send_message(message, inner_2)  # Player 2's choice

        # Step 3: Format the results
        result = f"Player 1: {response1.content}\nPlayer 2: {response2.content}\n"

        # Step 4: Create judgement prompt for the judge (LLM)
        judgement = f"{JUDGE}{result}Who wins?"
        message = TextMessage(content=judgement, source="user")

        # Step 5: Call the model (judge) to determine the winner
        # This is the key call to the LLM that evaluates the game
        response = await self._delegate.on_messages([message], ctx.cancellation_token)

        # Step 6: Return the complete result including the judge's decision
        return Message(content=result + response.chat_message.content)

In [ ]:
runtime = SingleThreadedAgentRuntime()
await Player1Agent.register(runtime, "player1", lambda: Player1Agent("player1"))
await Player2Agent.register(runtime, "player2", lambda: Player2Agent("player2"))
await RockPaperScissorsAgent.register(runtime, "rock_paper_scissors", lambda: RockPaperScissorsAgent("rock_paper_scissors"))
runtime.start()

INFO:autogen_core:Calling message handler for rock_paper_scissors/default with message type Message sent by Unknown
INFO:autogen_core.events:{"payload": "{\"content\": \"go\"}", "sender": null, "receiver": "rock_paper_scissors/default", "kind": "MessageKind.DIRECT", "delivery_stage": "DeliveryStage.DELIVER", "type": "Message"}
INFO:autogen_core.events:{"payload": "{\"content\": \"go\"}", "sender": null, "receiver": "rock_paper_scissors/default", "kind": "MessageKind.DIRECT", "delivery_stage": "DeliveryStage.DELIVER", "type": "Message"}
INFO:autogen_core.events:{"payload": "{\"content\": \"You are playing rock, paper, scissors. Respond only with the one word, one of the following: rock, paper, or scissors.\"}", "sender": "rock_paper_scissors/default", "receiver": "player1/default", "kind": "MessageKind.DIRECT", "delivery_stage": "DeliveryStage.SEND", "type": "Message"}
INFO:autogen_core.events:{"payload": "{\"content\": \"You are playing rock, paper, scissors. Respond only with the one 

In [21]:
agent_id = AgentId("rock_paper_scissors", "default")
message = Message(content="go")
response = await runtime.send_message(message, agent_id)
print(response.content)

INFO:autogen_core.events:{"payload": "{\"content\": \"go\"}", "sender": null, "receiver": "rock_paper_scissors/default", "kind": "MessageKind.DIRECT", "delivery_stage": "DeliveryStage.SEND", "type": "Message"}
INFO:autogen_core:Sending message of type Message to rock_paper_scissors: {'content': 'go'}
INFO:autogen_core:Sending message of type Message to rock_paper_scissors: {'content': 'go'}


Player 1: rock
Player 2: scissors
Player 1 wins, as rock crushes scissors. TERMINATE


In [22]:
await runtime.stop()
await runtime.close()